# Data Processing + Clean Up

### GEO ID EXTRACTION

In [38]:
import os
import fiona
import pandas as pd
import geopandas as gpd
print(os.getcwd())

/Users/michelenaorourke/Desktop/6.C51/1.C51/1.C51-Final-Project


In [39]:

# Check which layers are inside the file
gpkg = 'data/facilities.gpkg' # provide path to gpkg file
for layer in fiona.listlayers(gpkg): #iterate through each layer and read the columns within each layer
    gdf = gpd.read_file(filename=gpkg, layer=layer, rows=30)
    print(f"Layer: {layer}")
    print("Columns:")
    for colname, coltype in gdf.dtypes.to_dict().items():
        print(f"column name: {colname} data type: {coltype}")
    print(f"Geometry type: {gdf.geometry.iloc[0].geom_type}\n")

# Based on this dataset, it looks like there is only 1 layer, with multiple columns

Layer: facilities
Columns:
column name: facility_id data type: str
column name: facility_name data type: str
column name: facility_other_names data type: str
column name: sub_site_name data type: object
column name: sub_site_other_names data type: object
column name: facility_type data type: str
column name: primary_commodity data type: str
column name: commodities_products data type: str
column name: facility_equipment data type: str
column name: production_start data type: float64
column name: production_end data type: float64
column name: activity_status data type: str
column name: activity_status_year data type: float64
column name: surface_area_sq_km data type: float64
column name: concession_area_sq_km data type: float64
column name: country data type: str
column name: GID_0 data type: str
column name: GID_1 data type: str
column name: GID_2 data type: str
column name: GID_3 data type: str
column name: GID_4 data type: str
column name: source_id data type: str
column name: commen

In [40]:
# Read the full GeoPackage file
import geopandas as gpd

gdf_full = gpd.read_file('data/facilities.gpkg')

# Select the specified columns
columns = [
    'facility_id', 'facility_name', 'facility_other_names', 'sub_site_name', 
    'sub_site_other_names', 'facility_type', 'primary_commodity', 
    'commodities_products', 'facility_equipment', 'production_start', 
    'production_end', 'activity_status', 'activity_status_year', 
    'surface_area_sq_km', 'concession_area_sq_km', 'country'
]

df = gdf_full[columns].copy()

# Convert data types as specified
df = df.astype({
    'facility_id': 'object',
    'facility_name': 'object',
    'facility_other_names': 'object',
    'sub_site_name': 'object',
    'sub_site_other_names': 'object',
    'facility_type': 'object',
    'primary_commodity': 'object',
    'commodities_products': 'object',
    'facility_equipment': 'object',
    'production_start': 'float64',
    'production_end': 'float64',
    'activity_status': 'object',
    'activity_status_year': 'float64',
    'surface_area_sq_km': 'float64',
    'concession_area_sq_km': 'float64',
    'country': 'object'
})

print(f"DataFrame shape: {df.shape}")
print(df.dtypes)

DataFrame shape: (2413, 16)
facility_id               object
facility_name             object
facility_other_names      object
sub_site_name             object
sub_site_other_names      object
facility_type             object
primary_commodity         object
commodities_products      object
facility_equipment        object
production_start         float64
production_end           float64
activity_status           object
activity_status_year     float64
surface_area_sq_km       float64
concession_area_sq_km    float64
country                   object
dtype: object


In [41]:
df['country'].unique()

array(['Canada', 'India', 'United States of America', 'Kazakhstan',
       'Russian Federation', 'Argentina', 'Brazil', 'Spain', 'Ghana',
       'Mexico', 'Australia', 'Sweden', 'Saudi Arabia', 'Portugal',
       'Peru', 'Jamaica', 'Nigeria', 'Chile', 'Cameroon', 'South Africa',
       'Madagascar', 'United Kingdom', 'China', 'Germany', 'Poland',
       'Indonesia', 'Ireland', 'Ukraine', 'Mongolia', 'Guyana',
       'Luxembourg', 'Gabon', 'Mozambique', 'Guinea', "Cote d'Ivoire",
       'Kyrgyzstan', 'Tanzania', 'Zambia', 'Philippines', 'Venezuela',
       'Turkey', 'Colombia', 'Panama', 'Suriname', 'France', 'Guatemala',
       'Italy', 'Dominican Republic', 'Iceland', 'Zimbabwe', 'DR Congo',
       'Ecuador', 'United Arab Emirates', 'Romania', 'Belgium',
       'Mauritania', 'Finland', 'Papua New Guinea', 'New Caledonia',
       'Norway', 'Mali', 'New Zealand', 'Cuba', 'Fiji Islands', 'Namibia',
       'Czech Republic', 'Malaysia', 'Trinidad and Tobago',
       'Bosnia and Herzegovina

In [42]:
# Save the DataFrame to CSV
df.to_csv('data/facilities_data.csv', index=False)
print("DataFrame saved to facilities_data.csv")
print(len(df))
#things to do next:
#1. Check for missing feature values and handle them appropriately (e.g., imputation, removal
#2. add a column for the average country wage
#3. see what rows exist that have a production start date remove all other rows
# activity status year vs production start year


DataFrame saved to facilities_data.csv
2413


In [43]:
ids_no_start_date = df[df['production_start'].isnull()]['facility_id']
print("Facility IDs with missing production start dates:", len(ids_no_start_date))

df_no_start_date = df
df_no_start_date.dropna(subset=['production_start'], inplace=True) 
print(len(df))
df_no_start_date.to_csv('data/facilities_data_after_dropping_start_dates.csv', index=False)

Facility IDs with missing production start dates: 1815
598


In [ ]:
#Remove Production End Date, Activity Status Year, Activity Status, Surface Area, Concession Area 
df_no_start_date.drop(columns=['production_end', 'sub_site_name','sub_site_other_names', 'activity_status_year', 'activity_status', 'surface_area_sq_km', 'concession_area_sq_km'], inplace=True)
#Remove rows with facility_type that does not contain "mine"
df_no_start_date = df_no_start_date[df_no_start_date['facility_type'].str.contains('mine', case=False, na=False)]
print(len(df_no_start_date))
df_no_start_date.to_csv('data/facilities_data_after_dropping_start_dates_only_mines.csv', index=False)


534


In [45]:
df_no_start_date['country'].unique()

array(['United States of America', 'Spain', 'Ghana', 'Kazakhstan',
       'Saudi Arabia', 'Mexico', 'Russian Federation', 'Portugal',
       'Madagascar', 'Chile', 'Brazil', 'Peru', 'China', 'Australia',
       'Indonesia', 'South Africa', 'Mozambique', 'Canada', 'India',
       'Kyrgyzstan', 'Guinea', 'Colombia', 'Argentina', 'Zambia',
       'Suriname', 'Philippines', 'Guatemala', 'DR Congo', 'Ukraine',
       'Jamaica', 'Finland', 'Mali', 'Poland', 'New Zealand', 'Cuba',
       'Zimbabwe', 'Malaysia', 'Bosnia and Herzegovina',
       'Dominican Republic', 'Namibia', 'Bolivia', 'Papua New Guinea',
       "Cote d'Ivoire", 'Liberia', 'Tajikistan', 'Sweden'], dtype=object)